In [34]:
DATA = 'data.json'
URL = 'https://collectionapi.metmuseum.org/public/collection/v1/'

In [26]:
%pip install requests google-genai

3360.03s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


  Using cached google_genai-1.69.0-py3-none-any.whl.metadata (52 kB)
  Using cached anyio-4.13.0-py3-none-any.whl.metadata (4.5 kB)
  Using cached google_auth-2.49.1-py3-none-any.whl.metadata (6.2 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached pydantic-2.12.5-py3-none-any.whl.metadata (90 kB)
  Using cached tenacity-9.1.4-py3-none-any.whl.metadata (1.2 kB)
  Using cached websockets-16.0-cp314-cp314-manylinux1_x86_64.manylinux_2_28_x86_64.manylinux_2_5_x86_64.whl.metadata (6.8 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached distro-1.9.0-py3-none-any.whl.metadata (6.8 kB)
  Using cached sniffio-1.3.1-py3-none-any.whl.metadata (3.9 kB)
  Using cached pyasn1_modules-0.4.2-py3-none-any.whl.metadata (3.5 kB)
  Using cached cryptography-46.0.6-cp311-abi3-manylinux_2_34_x86_64.whl.metadata (5.7 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (

In [29]:
import requests
import json

def get_id_data():
    response = requests.get(f'{URL}/objects')
    dados = response.json()
    return dados['objectIDs']

def get_images(id):
    images = requests.get(f'{URL}/objects/{id}')
    link = images.json().get('primaryImage')
    return link


In [30]:
import random

ids = get_id_data()

image_valid = None
while not image_valid:
    id_random = random.choice(ids)
    image_valid = get_images(id_random)

print(image_valid)


https://images.metmuseum.org/CRDImages/dp/original/DP876658.jpg


In [ ]:
from google import genai
from google.genai import types
import config

client = genai.Client(api_key=config.GEMINI_API_KEY)

download = requests.get(image_valid)

mime_type = download.headers.get('Content-Type', 'image/jpeg').split(';')[0]

response = client.models.generate_content(
    model="gemini-3-flash-preview",
    contents=[
        types.Part.from_bytes(
            data=download.content,
            mime_type=mime_type
        ),
        "O que você vê nessa imagem?"
    ]
)

print(response.text)

A imagem é uma litografia satírica do famoso caricaturista francês **Honoré Daumier**, datada de 1857. Ela faz parte de uma série intitulada *"Croquis Parisiens"* (Esboços Parisienses).

Aqui está uma descrição detalhada do que se vê:

**1. A Cena Principal:**
No centro da imagem, uma mulher está em um estado de choque e pânico. Ela usa uma **crinolina** (uma armação de aros de metal usada sob as saias para dar volume), que era o auge da moda na época. O texto na parte inferior explica o que aconteceu: uma das molas de aço da sua anágua quebrou. 

A mola quebrada saltou para fora da saia de forma descontrolada, formando um grande arco metálico que se enrola em volta de um homem alto e magro que passava ao lado. O homem, que usa cartola e casaco comprido, parece surpreso e ligeiramente incomodado pela situação.

**2. Elementos de Humor e Sátira:**
*   **A expressão da mulher:** Seus braços estão abertos e sua boca escancarada, capturando o momento exato do acidente embaraçoso.
*   **O e

In [36]:
import os

new_data = {
    'id':id_random,
    'link': image_valid,
    'description': response.text
}

if os.path.exists(DATA):
    with open(DATA, 'r', encoding='utf-8') as f:
        try:
            existing_data = json.load(f)
        except json.JSONDecodeError:
            existing_data = []
else:
    existing_data = []

existing_data.append(new_data)

with open(DATA, 'w', encoding='utf-8') as f:
    json.dump(existing_data, f, indent=4, ensure_ascii=False)

print(f'ID: {id_random} e Link: {image_valid}')

ID: 678729 e Link: https://images.metmuseum.org/CRDImages/dp/original/DP876658.jpg
